# Kiểm kê dữ liệu và khóa nối

Notebook chứa mã phân tích theo đúng thứ tự từ các file nguồn trong `src/`. Chạy với kernel của `.venv` tại repo. Các bước có thể đọc toàn bộ raw và ghi vào `outputs/`; kiểm tra phạm vi và quy ước thời gian của từng phần trước khi diễn giải. Mã cũ dùng UTC hoặc mẫu là phương pháp cũ, chưa được coi là kết quả Moscow/toàn bộ khách hàng. Script nguồn vẫn được giữ để đối chiếu.

In [1]:
from pathlib import Path
import sys
import time
import pandas as pd
from IPython.display import display, Image, SVG, Markdown
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT / 'src/scripts'))
sys.path.insert(0, str(ROOT / 'src/mbd_mini_eda_fixes'))
def show_generated_results(started_ns, source):
    changed = sorted((p for base in (ROOT / 'outputs', ROOT / 'docs') if base.exists() for p in base.rglob('*') if p.is_file() and p.stat().st_mtime_ns >= started_ns), key=lambda p: p.as_posix())
    display(Markdown(f'### Kết quả: `{source}`'))
    if not changed:
        display(Markdown('Không có file kết quả mới từ phần này.'))
        return
    images = [p for p in changed if p.suffix.lower() in ('.png', '.jpg', '.jpeg', '.svg')]
    png_stems = {p.with_suffix('') for p in images if p.suffix.lower() == '.png'}
    for p in images:
        if p.suffix.lower() == '.svg' and p.with_suffix('') in png_stems:
            continue
        display(Markdown(f'**{p.relative_to(ROOT).as_posix()}**'))
        display(SVG(filename=str(p)) if p.suffix.lower() == '.svg' else Image(filename=str(p)))
    tables = [p for p in changed if p.suffix.lower() == '.csv']
    for p in tables:
        display(Markdown(f'**{p.relative_to(ROOT).as_posix()}** — 5 dòng đầu'))
        try:
            display(pd.read_csv(p, nrows=5))
        except Exception as exc:
            display(Markdown(f'Không đọc được bảng: `{type(exc).__name__}: {exc}`'))
    others = [p.relative_to(ROOT).as_posix() for p in changed if p not in images and p not in tables]
    if others:
        display(Markdown('File kết quả khác: ' + ', '.join(f'`{p}`' for p in others)))

## Audit Mbd Inventory

Nguồn: `src/audit_mbd_inventory.py`. Chạy các ô của phần này theo thứ tự.

In [2]:
__file__ = str(ROOT / 'src/audit_mbd_inventory.py')
sys.argv = [__file__]
__eda_started_ns = time.time_ns()
__name__ = '__main__'
__package__ = None


In [3]:
r"""Count every MBD-mini Parquet footer and inspect all top-level schemas.

Writes aggregate inventory only; source files are read without modification.
Run: .\.venv\Scripts\python.exe -X utf8 src/audit_mbd_inventory.py
"""

from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
import json
from pathlib import Path
from zoneinfo import ZoneInfo

import pyarrow.parquet as pq


ROOT = Path(__file__).resolve().parents[1]
BASE = ROOT / "data" / "MBD-mini"
OUT = ROOT / "docs" / "mbd_inventory"
TABLES = {
    "client_split": "Khách hàng",
    "targets": "Khách hàng–tháng",
    "detail/trx": "Sự kiện giao dịch",
    "detail/geo": "Sự kiện vị trí",
    "detail/dialog": "Sự kiện hội thoại",
    "ptls/trx": "Chuỗi giao dịch của khách",
    "ptls/geo": "Chuỗi vị trí của khách",
    "ptls/dialog": "Chuỗi hội thoại của khách",
}


def inspect_file(path):
    pf = pq.ParquetFile(path)
    fields = tuple((f.name, str(f.type)) for f in pf.schema_arrow)
    partitions = tuple(sorted({p.split("=", 1)[0] for p in path.relative_to(BASE).parts[:-1] if "=" in p}))
    return pf.metadata.num_rows, fields, partitions


def main():
    tables = {}
    all_columns = set()
    for name, unit in TABLES.items():
        files = sorted((BASE / name).rglob("*.parquet"))
        if not files:
            raise ValueError(f"No Parquet files: {name}")
        with ThreadPoolExecutor(max_workers=8) as pool:
            results = list(pool.map(inspect_file, files))
        schemas = {r[1] for r in results}
        partitions = {r[2] for r in results}
        if len(schemas) != 1 or len(partitions) != 1:
            raise ValueError(f"Inconsistent schemas/partition keys: {name}")
        fields = next(iter(schemas))
        partition_keys = list(next(iter(partitions)))
        physical = [n for n, _ in fields]
        logical = list(dict.fromkeys(physical + partition_keys))
        all_columns.update(logical)
        tables[name] = {
            "files": len(files), "rows": sum(r[0] for r in results), "unit": unit,
            "physical_column_count": len(physical), "logical_column_count": len(logical),
            "physical_columns": dict(fields), "partition_keys": partition_keys,
        }
        print(name, tables[name]["rows"], len(physical), len(logical), flush=True)

    clients = pq.read_table(BASE / "client_split", columns=["client_id"]).to_pandas()
    targets = pq.read_table(BASE / "targets", columns=["client_id", "mon"]).to_pandas()
    if len(clients) != tables["client_split"]["rows"] or len(targets) != tables["targets"]["rows"]:
        raise ValueError("Footer and read row counts differ")
    summary = {
        "checked_at": datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).isoformat(),
        "scope": "Every Parquet footer and top-level schema; full client_split IDs and target client-month keys",
        "tables": tables,
        "parquet_files": sum(t["files"] for t in tables.values()),
        "unique_clients": int(clients.client_id.nunique()),
        "target_unique_clients": int(targets.client_id.nunique()),
        "duplicate_client_ids_in_split": int(clients.client_id.duplicated().sum()),
        "duplicate_target_client_months": int(targets.duplicated(["client_id", "mon"]).sum()),
        "months": sorted(targets.mon.unique().tolist()),
        "detail_events": sum(t["rows"] for n, t in tables.items() if n.startswith("detail/")),
        "distinct_top_level_column_names": sorted(all_columns),
    }
    OUT.mkdir(parents=True, exist_ok=True)
    (OUT / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    lines = [
        "# Kiểm đếm biến và quan sát MBD-mini", "",
        f"Thời điểm: {summary['checked_at']}. Quét đủ {summary['parquet_files']:,} file Parquet; kiểm tra schema của từng file và số dòng trong footer. Đọc toàn bộ khóa khách hàng–tháng để kiểm tra số khách, tháng và trùng khóa.", "",
        "| Bảng | Quan sát (dòng) | Đơn vị quan sát | Cột trong file | Cột khi gồm partition |",
        "|---|---:|---|---:|---:|",
    ]
    for name, t in tables.items():
        lines.append(f"| `{name}` | {t['rows']:,} | {t['unit']} | {t['physical_column_count']} | {t['logical_column_count']} |")
    lines += ["", f"Có **{summary['unique_clients']:,} khách hàng**, **{len(summary['months'])} tháng nhãn**, **{tables['targets']['rows']:,} mẫu khách hàng–tháng** và **{summary['detail_events']:,} sự kiện detail**.", "",
        "`fold` được lưu trong đường dẫn partition, nên số cột khi đọc cả bảng nhiều hơn số cột vật lý trong file một cột. `embedding` được tính là một cột mảng; theo audit nội dung trước, mỗi vector hội thoại có 768 phần tử. Các cột mảng trong ptls cũng được tính là một cột mỗi trường, không trải phẳng thành số phần tử.", "",
        "Không cộng detail và ptls thành tổng quan sát độc lập: đó là hai dạng biểu diễn của lịch sử. Cũng không cộng các bảng có đơn vị quan sát khác nhau để suy ra số mẫu train. Số cột nguồn chưa phải số feature sau tổng hợp lịch sử.", "",
        f"Không trùng ID trong client_split: {summary['duplicate_client_ids_in_split'] == 0}. Không trùng khóa khách hàng–tháng trong targets: {summary['duplicate_target_client_months'] == 0}.", "",
        "Tên cột và kiểu dữ liệu của từng bảng nằm trong `summary.json`. Chạy lại: `.\\.venv\\Scripts\\python.exe -X utf8 src/audit_mbd_inventory.py`.", ""]
    (OUT / "assessment.md").write_text("\n".join(lines), encoding="utf-8")
    print(json.dumps({k: v for k, v in summary.items() if k != "tables"}, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()

client_split 100224 1 2


targets 1202688 8 9


detail/trx 38764930 14 15


detail/geo 66295724 5 6


detail/dialog 217205 3 4


ptls/trx 98721 14 15


ptls/geo 72573 5 6


ptls/dialog 46320 3 4


{
  "checked_at": "2026-11-19T14:42:17.064818+07:00",
  "scope": "Every Parquet footer and top-level schema; full client_split IDs and target client-month keys",
  "parquet_files": 3820,
  "unique_clients": 100224,
  "target_unique_clients": 100224,
  "duplicate_client_ids_in_split": 0,
  "duplicate_target_client_months": 0,
  "months": [
    "2022-02-28",
    "2022-03-31",
    "2022-04-30",
    "2022-05-31",
    "2022-06-30",
    "2022-07-31",
    "2022-08-31",
    "2022-09-30",
    "2022-10-31",
    "2022-11-30",
    "2022-12-31",
    "2023-01-31"
  ],
  "detail_events": 105277859,
  "distinct_top_level_column_names": [
    "amount",
    "client_id",
    "currency",
    "diff_trans_date",
    "dst_type11",
    "dst_type12",
    "embedding",
    "event_subtype",
    "event_time",
    "event_type",
    "fold",
    "geohash_4",
    "geohash_5",
    "geohash_6",
    "mon",
    "src_type11",
    "src_type12",
    "src_type21",
    "src_type22",
    "src_type31",
    "src_type32",
    "tar

In [4]:
show_generated_results(__eda_started_ns, 'src/audit_mbd_inventory.py')

### Kết quả: `src/audit_mbd_inventory.py`

File kết quả khác: `docs/mbd_inventory/assessment.md`, `docs/mbd_inventory/summary.json`

## Audit Mbd Join Questions

Nguồn: `src/audit_mbd_join_questions.py`. Chạy các ô của phần này theo thứ tự.

In [5]:
__file__ = str(ROOT / 'src/audit_mbd_join_questions.py')
sys.argv = [__file__]
__eda_started_ns = time.time_ns()
__name__ = '__main__'
__package__ = None


In [6]:
"""Recompute join, full-history presence and temporal diagnostics; aggregates only."""
from pathlib import Path
import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

ROOT = Path(__file__).resolve().parents[1]
BASE = ROOT / 'data/MBD-mini'

def main():
    clients = pq.read_table(BASE / 'client_split').to_pandas().sort_values('client_id').reset_index(drop=True)
    targets = pq.read_table(BASE / 'targets').to_pandas()
    ids = pd.Index(clients.client_id)
    n = len(ids)
    months = sorted(targets.mon.unique())
    bounds = pd.to_datetime(months).to_numpy(dtype='datetime64[ns]').astype('int64')
    last = np.full(n, np.iinfo('int64').min, dtype='int64')
    after = np.zeros((n, len(months)), dtype='int64')
    presence = {}
    total = 0
    for mod in ('trx', 'geo', 'dialog'):
        seen = np.zeros(n, dtype=bool)
        for path in (BASE / 'detail' / mod).rglob('*.parquet'):
            cols = ['client_id', 'event_time'] if mod == 'trx' else ['client_id']
            frame = pq.ParquetFile(path).read(columns=cols).to_pandas()
            pos = ids.get_indexer(frame.client_id)
            assert (pos >= 0).all()
            seen[pos] = True
            if mod == 'trx':
                assert frame.event_time.notna().all()
                times = frame.event_time.to_numpy(dtype='datetime64[ns]').astype('int64')
                np.maximum.at(last, pos, times)
                total += len(frame)
                for j, bound in enumerate(bounds):
                    after[:, j] += np.bincount(pos[times > bound], minlength=n)
        presence[mod] = seen
        print(mod, int(seen.sum()), flush=True)
    has_trx = presence['trx']
    delta = np.full((n, len(months)), np.nan)
    delta[has_trx] = np.floor((bounds[None, :] - last[has_trx, None]) / 86400e9)
    labels = [f'target_{i}' for i in range(1, 5)]
    y = targets[labels].eq(1)
    folds = []
    for fold, group in clients.groupby('fold', observed=True):
        ix = group.index.to_numpy()
        folds.append({'fold': int(fold), 'clients': len(ix), **{f'{m}_absent_percent': float((~p[ix]).mean()*100) for m,p in presence.items()}})
    monthly = []
    for j, month in enumerate(months):
        d = delta[:, j]
        monthly.append({'mon': str(month), 'clients_any_trx_after_mon': int((after[:, j]>0).sum()),
                        'percent_all_clients_any_after_mon': float((after[:,j]>0).mean()*100),
                        'transactions_after_mon': int(after[:,j].sum()),
                        'percent_transactions_after_mon': float(after[:,j].sum()/total*100),
                        'delta_min_median_max_days': [float(np.nanmin(d)),float(np.nanmedian(d)),float(np.nanmax(d))],
                        'full_history_last_trx_more_than_180_days_before_mon': int((d>180).sum()),
                        'full_history_last_trx_more_than_365_days_before_mon': int((d>365).sum())})
    valid = delta[np.isfinite(delta)]
    extremes = {}
    for name, mask in {'min':delta==valid.min(),'max':delta==valid.max(),'gt180':delta>180,'gt365':delta>365,'lt_minus180':delta < -180}.items():
        extremes[name] = {'client_months':int(mask.sum()),'unique_clients':int(mask.any(axis=1).sum())}
    joined = targets.merge(clients, on='client_id', validate='many_to_one', suffixes=('_target','_split'))
    assert len(joined)==len(targets)
    assert joined.fold_target.eq(joined.fold_split).all()
    result = {'definitions': {'presence':'ID occurs in detail over ALL available history, no attribute filtering',
                             'after_mon':'event_time > mon at 00:00:00; same-day later events included',
                             'delta':'floor((mon - full-history last_trx_time) / 1 day); NOT safe training recency',
                             'event_denominator':'all detail/trx rows; computed independently per mon, not summed over months'},
              'clients':n,'target_rows':len(targets),'duplicate_client_months':int(targets.duplicated(['client_id','mon']).sum()),
              'months_per_client_counts':{str(k):int(v) for k,v in targets.groupby('client_id').mon.nunique().value_counts().items()},
              'folds':folds, 'transaction_rows':total,'monthly':monthly,
              'delta_quantiles_days':{str(q):float(np.quantile(valid,q)) for q in (0,.01,.05,.25,.5,.75,.95,.99,1)},
              'delta_extremes':extremes,
              'positive_counts':{c:int(y[c].sum()) for c in labels},
              'positive_percent':{c:float(y[c].mean()*100) for c in labels},
              'positive_label_count_distribution':{str(k):int(v) for k,v in y.sum(axis=1).value_counts().sort_index().items()},
              'pair_overlap_same_client_month':{a+' & '+b:int((y[a]&y[b]).sum()) for i,a in enumerate(labels) for b in labels[i+1:]}}
    out = ROOT / 'docs/propensity_feasibility/join_questions_audit.json'
    out.write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
    print(json.dumps(result,ensure_ascii=False,indent=2),flush=True)

if __name__ == '__main__':
    main()

trx 98721


geo 72573


dialog 46320


{
  "definitions": {
    "presence": "ID occurs in detail over ALL available history, no attribute filtering",
    "after_mon": "event_time > mon at 00:00:00; same-day later events included",
    "delta": "floor((mon - full-history last_trx_time) / 1 day); NOT safe training recency",
    "event_denominator": "all detail/trx rows; computed independently per mon, not summed over months"
  },
  "clients": 100224,
  "target_rows": 1202688,
  "duplicate_client_months": 0,
  "months_per_client_counts": {
    "12": 100224
  },
  "folds": [
    {
      "fold": 0,
      "clients": 20328,
      "trx_absent_percent": 1.4561196379378198,
      "geo_absent_percent": 26.88410074773711,
      "dialog_absent_percent": 53.7534435261708
    },
    {
      "fold": 1,
      "clients": 19902,
      "trx_absent_percent": 1.5274846749070445,
      "geo_absent_percent": 27.600241181790775,
      "dialog_absent_percent": 54.01969651291327
    },
    {
      "fold": 2,
      "clients": 19917,
      "trx_absent_

In [7]:
show_generated_results(__eda_started_ns, 'src/audit_mbd_join_questions.py')

### Kết quả: `src/audit_mbd_join_questions.py`

**docs/mbd_temporal_coverage/full_history_missingness.csv** — 5 dòng đầu

,metric,clients,opportunities_per_client,clients_no_events,clients_absent_strictly_over_50_percent,percent_clients_absent_strictly_over_50_percent,clients_at_least_50_percent_observed
0,any_source_active_days_all_initial_clients,100224,354,1156,42320,42.225415,57904
1,any_source_active_days_among_clients_with_any_...,99068,354,0,41164,41.551258,57904
2,any_source_active_days_among_all_three_source_...,25695,354,0,4854,18.890835,20841
3,source_day_cells_all_initial_clients,100224,1062,1156,95227,95.014168,4997
4,sources_available_all_initial_clients,100224,3,1156,22329,22.279095,77895


File kết quả khác: `docs/mbd_temporal_coverage/full_history_missingness.json`, `docs/propensity_feasibility/join_questions_audit.json`